# Part 1 - Ingesting Archive API data

## Downloading Packages, Connecting to MongoDB, & Connecting to API Key

In [1]:
# Downloading all necessary packages
import requests
import time
from pymongo import MongoClient

In [2]:
# Connecting to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["literary_trends"]
collection = db["critics_picks"]

In [3]:
# Connecting to API Key and fetching function
API_KEY = "0ohuAkd8bXkaTjGXRNFAq2IVzlwS1C2cVqquOvmSfZORMbzy"

def fetch_archive(year, month):
    url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json?api-key={API_KEY}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Failed: {year}-{month} | Status: {response.status_code}")
        return None

In [4]:
# Testing to ensure that connection worked
data = fetch_archive(2000, 1)
if data:
    articles = data.get("response", {}).get("docs", [])
    print(f"Articles fetched: {len(articles)}")
    print(articles[0])

Articles fetched: 8502
{'abstract': 'Article on upcoming New York Giants-Dallas Cowboys game; photo (M)', 'web_url': 'https://www.nytimes.com/2000/01/01/sports/pro-football-playoffs-or-no-dallas-provides-the-motivation.html', 'snippet': 'Article on upcoming New York Giants-Dallas Cowboys game; photo (M)', 'lead_paragraph': 'Waiting in the visiting locker room at Texas Stadium late tomorrow afternoon, the Giants will know whether the Green Bay Packers, who play earlier against Arizona, have won or are comfortably ahead.', 'print_section': 'D', 'print_page': '2', 'source': 'The New York Times', 'multimedia': [], 'headline': {'main': 'Playoffs or No, Dallas Provides The Motivation', 'kicker': 'PRO FOOTBALL', 'content_kicker': None, 'print_headline': 'PRO FOOTBALL; Playoffs or No, Dallas Provides The Motivation', 'name': None, 'seo': None, 'sub': None}, 'keywords': [{'name': 'organizations', 'value': 'New York Giants', 'rank': 1, 'major': 'N'}, {'name': 'organizations', 'value': 'Dallas Co

## Inserting Data into MongoDB & Verification

In [5]:
# Looping through 2000-2026 & Inserting into MongoDB
for year in range(2000, 2027):
    for month in range(1, 13):
        print(f"Fetching {year}-{month}...")
        data = fetch_archive(year, month)
        if data:
            articles = data.get("response", {}).get("docs", [])
            critics = [a for a in articles if a.get("type_of_material") == "Review"]
            if critics:
                collection.insert_many(critics)
                print(f"Inserted {len(critics)} Critics' Pick reviews")
        time.sleep(12)

# Verifying that data has been inserted into MongoDB
print(f"Total documents in MongoDB: {collection.count_documents({})}")

Fetching 2000-1...
Inserted 509 Critics' Pick reviews
Fetching 2000-2...
Inserted 503 Critics' Pick reviews
Fetching 2000-3...
Inserted 600 Critics' Pick reviews
Fetching 2000-4...
Inserted 546 Critics' Pick reviews
Fetching 2000-5...
Inserted 504 Critics' Pick reviews
Fetching 2000-6...
Inserted 465 Critics' Pick reviews
Fetching 2000-7...
Inserted 457 Critics' Pick reviews
Fetching 2000-8...
Inserted 376 Critics' Pick reviews
Fetching 2000-9...
Inserted 455 Critics' Pick reviews
Fetching 2000-10...
Inserted 576 Critics' Pick reviews
Fetching 2000-11...
Inserted 492 Critics' Pick reviews
Fetching 2000-12...
Inserted 576 Critics' Pick reviews
Fetching 2001-1...
Inserted 423 Critics' Pick reviews
Fetching 2001-2...
Inserted 459 Critics' Pick reviews
Fetching 2001-3...
Inserted 475 Critics' Pick reviews
Fetching 2001-4...
Inserted 462 Critics' Pick reviews
Fetching 2001-5...
Inserted 427 Critics' Pick reviews
Fetching 2001-6...
Inserted 445 Critics' Pick reviews
Fetching 2001-7...
Insert

# Part 2 - Ingesting Books API Data

## Connecting to MongoDB, & Connecting to API Key

In [14]:
# Downloading further necessary packages
from datetime import date, timedelta

# Connecting to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["literary_trends"]
collection = db["bestsellers"]

In [15]:
# Connecting to API Key and fetching function
API_KEY = "0ohuAkd8bXkaTjGXRNFAq2IVzlwS1C2cVqquOvmSfZORMbzy"

def fetch_bestsellers(list_name, date):
    url = f"https://api.nytimes.com/svc/books/v3/lists/{date}/{list_name}.json?api-key={API_KEY}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Failed: {list_name} {date} | Status: {response.status_code}")
        return None

In [23]:
# Testing to ensure that connection worked
start_date = date(2016, 1, 1)
end_date = date(2026, 1, 1)
current_date = start_date

while current_date <= end_date:
    date_str = current_date.strftime("%Y-%m-%d")
    for list_name in ["hardcover-fiction", "hardcover-nonfiction"]:
        data = fetch_bestsellers(list_name, date_str)
        if data:
            books = data.get("results", {}).get("books", [])
            if books:
                for book in books:
                    book["list_name"] = list_name
                    book["list_date"] = date_str
                collection.insert_many(books)
                print(f"Inserted {len(books)} books from {list_name} {date_str}")
    current_date += timedelta(weeks=1)
    time.sleep(20)

print(f"Total documents in MongoDB: {collection.count_documents({})}")
print("Sample document:")
print(collection.find_one())

Inserted 20 books from hardcover-fiction 2016-01-01
Inserted 20 books from hardcover-nonfiction 2016-01-01
Inserted 20 books from hardcover-fiction 2016-01-08
Inserted 20 books from hardcover-nonfiction 2016-01-08
Inserted 20 books from hardcover-fiction 2016-01-15
Inserted 20 books from hardcover-nonfiction 2016-01-15
Inserted 20 books from hardcover-fiction 2016-01-22
Failed: hardcover-nonfiction 2016-01-22 | Status: 429
Inserted 20 books from hardcover-fiction 2016-01-29
Inserted 20 books from hardcover-nonfiction 2016-01-29
Inserted 20 books from hardcover-fiction 2016-02-05
Inserted 20 books from hardcover-nonfiction 2016-02-05
Inserted 20 books from hardcover-fiction 2016-02-12
Failed: hardcover-nonfiction 2016-02-12 | Status: 429
Inserted 20 books from hardcover-fiction 2016-02-19
Inserted 20 books from hardcover-nonfiction 2016-02-19
Inserted 20 books from hardcover-fiction 2016-02-26
Inserted 20 books from hardcover-nonfiction 2016-02-26
Inserted 20 books from hardcover-fictio

In [24]:
last = collection.find_one(sort=[("list_date", -1)])
print(last["list_date"])

2021-08-20
